In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("..").resolve()))

from src import banco
from src.limpeza import texto_para_data, texto_para_decimal, executar_limpeza

conexao = banco.conectar()

def consultar(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conexao)

print("Conectado ao banco com sucesso.")


Antes / depois: exemplos de conversão de tipo

In [ ]:
exemplos_valor = ["1.272,97", "0,00", "", "Sem informação"]
pd.DataFrame({
    "valor_bruto (Raw)": exemplos_valor,
    "valor_convertido (Silver)": [texto_para_decimal(v) for v in exemplos_valor],
})


In [ ]:
exemplos_data = ["01/03/2025", "04/08/2025", "", "31/02/2025"]
pd.DataFrame({
    "data_bruta (Raw)": exemplos_data,
    "data_convertida (Silver)": [texto_para_data(v) for v in exemplos_data],
})


Note que `"31/02/2025"` (31 de fevereiro não existe) e valores vazios se
tornam `None` — a função nunca lança exceção, apenas descarta o valor
inválido, para que uma linha ruim não interrompa a carga do lote inteiro.

Executando a transformação completa 

In [ ]:
executar_limpeza()


Validação da camada Silver

In [ ]:
tabelas_silver = ["silver_viagem", "silver_pagamento", "silver_passagem", "silver_trecho"]
volumes = {t: consultar(f"SELECT COUNT(*) AS total FROM {t}")["total"][0] for t in tabelas_silver}
pd.Series(volumes, name="linhas").to_frame()


Integridade referencial — não deve haver registro órfão

In [ ]:
sql_orfaos = """
    SELECT
        (SELECT COUNT(*) FROM silver_pagamento p
            LEFT JOIN silver_viagem v ON v.id_viagem = p.id_viagem
            WHERE v.id_viagem IS NULL) AS pagamentos_orfaos,
        (SELECT COUNT(*) FROM silver_passagem pa
            LEFT JOIN silver_viagem v ON v.id_viagem = pa.id_viagem
            WHERE v.id_viagem IS NULL) AS passagens_orfas,
        (SELECT COUNT(*) FROM silver_trecho t
            LEFT JOIN silver_viagem v ON v.id_viagem = t.id_viagem
            WHERE v.id_viagem IS NULL) AS trechos_orfaos;
"""
consultar(sql_orfaos)


Amostra da tabela `silver_viagem` já tipada

In [ ]:
consultar("""
    SELECT id_viagem, nome_orgao_superior, data_inicio, data_fim,
           duracao_dias, valor_total
    FROM silver_viagem
    LIMIT 5;
""")


In [ ]:
conexao.close()
print("Conexao encerrada.")
